In [ ]:
"""
Concepts:

1. Bagging
2. Bootstrap Sampling
3. OOB Score
4. Feature Importance
5. Permutation Importance
6. Variance Reduction
7. Hyperparameter Tuning
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
from sklearn.inspection import permutation_importance

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

housing = fetch_california_housing()
df = pd.DataFrame(housing.data,columns=housing.feature_names)
df["Target"] = housing.target
print(df.head())
print(df.info())
print(df.describe())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())
print(df.corr())

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("Target", axis=1)
y = df["Target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# Trees do not require scaling

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

rfr = RandomForestRegressor(n_estimators=100,bootstrap=True,oob_score=True,n_jobs=-1,random_state=42)
rfr.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : METRICS for prediction against validation data with base model
# =====================================================

y_pred = rfr.predict(X_val)
mae = mean_absolute_error(y_val,y_pred)
rmse = np.sqrt(mean_squared_error(y_val,y_pred))
r2 = r2_score(y_val,y_pred)

print("\nBASELINE RANDOM FOREST")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2 :", r2)


In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(rfr,X_train,y_train,cv=5,scoring="r2",n_jobs=-1)
print("\nCV Mean:", cv_scores.mean())
print("CV Std :", cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : GRID SEARCH
# =====================================================

param_grid = {
    "n_estimators":[100,200,300],
    "max_depth":[5,10,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,5],
    "max_features":["sqrt","log2"]}

grid = GridSearchCV(RandomForestRegressor(random_state=42,n_jobs=-1),param_grid,cv=3,scoring="r2",n_jobs=-1)
grid.fit(X_train,y_train)
print(grid.best_params_)
best_model = grid.best_estimator_

# =====================================================
# STEP 8.5 : RANDOMIZED SEARCH
# =====================================================

random_search = RandomizedSearchCV(RandomForestRegressor(random_state=42,n_jobs=-1),
                                   param_gridcv=3,n_iter=15,scoring="r2",random_state=42,n_jobs=-1)

random_search.fit(X_train,y_train)
print(random_search.best_params_)


In [ ]:
# =====================================================
# STEP 9 : METRICS for prediction against validation data with best model
# =====================================================

y_best = best_model.predict(X_val)
val_r2 = r2_score(y_val,y_best)
print("Validation R2:", val_r2)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION with best model
# =====================================================

y_train_best = best_model.predict(X_train)
train_r2 = r2_score(y_train,y_train_best)

In [ ]:
# =====================================================
# STEP 11 : RANDOM FOREST ANALYSIS
# =====================================================

# ---------------------------------------------
# OOB SCORE
# ---------------------------------------------
print(rfr.oob_score_)
# ---------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_}).sort_values(by="Importance",ascending=False)
print(importance_df)
plt.figure(figsize=(8,5))
plt.barh(importance_df["Feature"],importance_df["Importance"])
plt.title("Feature Importance")
plt.show()

# ---------------------------------------------
# PERMUTATION IMPORTANCE
# ---------------------------------------------

perm = permutation_importance(best_model,X_val,y_val,n_repeats=10,random_state=42)
perm_df = pd.DataFrame({"Feature":X.columns,"Importance":perm.importances_mean}).sort_values(by="Importance",ascending=False)
print(perm_df)

# ---------------------------------------------
# Number of Trees Analysis
# ---------------------------------------------

tree_counts = [10,50,100,200,300]
scores = []

for trees in tree_counts:
    model = RandomForestRegressor(n_estimators=trees,random_state=42,n_jobs=-1)
    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    scores.append(score)

plt.plot(tree_counts,scores,marker="o")
plt.xlabel("Trees")
plt.ylabel("Validation R2")
plt.title("n_estimators Analysis")
plt.show()

# ---------------------------------------------
# Decision Tree vs Random Forest
# ---------------------------------------------

from sklearn.tree import DecisionTreeRegressor
dtr = DecisionTreeRegressor(random_state=42)
dtr.fit(X_train,y_train)
dt_pred = dtr.predict(X_val)
dt_r2 = r2_score(y_val,dt_pred)
rf_r2 = r2_score(y_val,best_model.predict(X_val))
print("\nDecision Tree R2:", dt_r2)
print("Random Forest R2:", rf_r2)


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test,test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test,test_pred))
test_r2 = r2_score(y_test,test_pred)
print("\nTEST RESULTS")
print("MAE :", test_mae)
print("RMSE:", test_rmse)
print("R2 :", test_r2)


In [ ]:
# =====================================================
# STEP 14 : INTERPRETATION
# =====================================================

print("\nTOP FEATURES")
print(importance_df.head(10))

In [ ]:
# =====================================================
# STEP 15 : DEPLOYMENT READINESS
# =====================================================

print("\nDEPLOYMENT READINESS")
print("CV Mean:",cv_scores.mean())
print("Validation R2:",val_r2)
print("Test R2:",test_r2)


# =====================================================
# INTERVIEW QUESTIONS
# =====================================================

"""
1. What is Random Forest?
2. Why is it called a forest?
3. What is bagging?
4. What is bootstrap sampling?
5. What is OOB score?
6. Difference between Bagging and Random Forest?
7. Why random feature selection?
8. What is max_features?
9. Why Random Forest reduces variance?
10. Why does Random Forest overfit less than trees?
11. What is feature importance?
12. Difference between Feature Importance
    and Permutation Importance?
13. What is n_estimators?
14. Can Random Forest overfit?
15. Why doesn't Random Forest need scaling?
16. When should you use Random Forest?
17. Random Forest vs XGBoost?
18. Random Forest vs Decision Tree?
19. Why is Random Forest parallelizable?
20. What are the disadvantages of Random Forest?
"""